# Проект по прогнозированию стоимости автомобиля

Это завершенная версия исходного ноутбука `Car_prices.ipynb`. В ней сохранена логика проекта и использован тот же датасет Kaggle: [Old Car Price Prediction](https://www.kaggle.com/datasets/milanvaddoriya/old-car-price-prediction).

Задача - регрессия: по характеристикам автомобиля предсказать цену в индийских рупиях. Дополнительно нужно понять, какие признаки важны для модели, а какие дают слабый вклад.

## План работы

1. Загрузить данные.
2. Очистить текстовые числовые поля: цену, пробег, объем двигателя, количество мест.
3. Провести EDA и построить графики.
4. Подготовить признаки для модели.
5. Обучить baseline и финальную модель.
6. Оценить качество через MAE, RMSE, R2.
7. Проанализировать важные и неважные признаки.
8. Сохранить готовую модель.

In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    import joblib
except ImportError:
    joblib = None

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 120)

RANDOM_STATE = 42
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

## 1. Загрузка данных

В исходном ноутбуке данные загружались из локального пути `/Users/salux/Downloads/car_price.csv`. Для воспроизводимости путь сделан относительным: положите файл `car_price.csv` в корень репозитория или в папку `data/`.

In [ ]:
DATA_PATHS = [
    Path("car_price.csv"),
    Path("data/car_price.csv"),
    Path("/Users/salux/Downloads/car_price.csv"),
]

dataset_path = next((path for path in DATA_PATHS if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError(
        "Не найден car_price.csv. Скачайте датасет Kaggle old-car-price-prediction "
        "и положите файл в корень репозитория или в папку data/."
    )

df_raw = pd.read_csv(dataset_path)
df = df_raw.copy()

print("Файл:", dataset_path)
print("Размер данных:", df.shape)
display(df.head(10))

## 2. Первичная проверка

В данных 10 колонок. Часть числовых признаков записана как текст: `10.03 Lakh`, `86,226 kms`, `1956 cc`, `5 Seats`. Перед моделированием эти поля нужно привести к числам.

In [ ]:
display(df.info())
display(df.describe(include="all").T)

missing = df.isna().mean().mul(100).sort_values(ascending=False).rename("missing_percent").to_frame()
display(missing)

print("Дубликаты строк:", df.duplicated().sum())

In [ ]:
plt.figure(figsize=(9, 4))
if (missing["missing_percent"] > 0).any():
    sns.barplot(data=missing.reset_index(), x="missing_percent", y="index", color="#4C78A8")
    plt.xlabel("Пропуски, %")
    plt.ylabel("Колонка")
else:
    plt.text(0.5, 0.5, "Пропусков нет", ha="center", va="center", fontsize=14)
    plt.axis("off")
plt.title("Доля пропусков по колонкам")
plt.tight_layout()
plt.show()

## 3. Очистка и создание признаков

Исправляем главные проблемы исходных данных:

- `car_prices_in_rupee` переводим в число рупий: `Lakh = 100000`, `Crore = 10000000`;
- `kms_driven`, `engine`, `Seats` очищаем от текста;
- из `ownership` выделяем номер владельца;
- из `car_name` выделяем бренд автомобиля;
- удаляем техническую колонку `Unnamed: 0`.

In [ ]:
def parse_price_rupee(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower().replace(",", "")
    match = re.search(r"([0-9]*\.?[0-9]+)", text)
    if not match:
        return np.nan
    number = float(match.group(1))
    if "crore" in text:
        return number * 10_000_000
    if "lakh" in text:
        return number * 100_000
    return number


def parse_first_number(value):
    if pd.isna(value):
        return np.nan
    text = str(value).replace(",", "")
    match = re.search(r"([0-9]*\.?[0-9]+)", text)
    return float(match.group(1)) if match else np.nan


def parse_ownership(value):
    if pd.isna(value):
        return np.nan
    text = str(value).lower()
    if "first" in text or "1st" in text:
        return 1
    if "second" in text or "2nd" in text:
        return 2
    if "third" in text or "3rd" in text:
        return 3
    if "fourth" in text or "4th" in text:
        return 4
    if "fifth" in text or "5th" in text:
        return 5
    return parse_first_number(text)


data = df.copy()
data = data.drop(columns=[col for col in ["Unnamed: 0"] if col in data.columns])
data = data.drop_duplicates()

data["price_rupee"] = data["car_prices_in_rupee"].apply(parse_price_rupee)
data["kms_driven_num"] = data["kms_driven"].apply(parse_first_number)
data["engine_cc"] = data["engine"].apply(parse_first_number)
data["seats_num"] = data["Seats"].apply(parse_first_number)
data["owner_number"] = data["ownership"].apply(parse_ownership)
data["brand"] = data["car_name"].astype(str).str.split().str[0]
data["car_age"] = 2026 - data["manufacture"]

data = data.dropna(subset=["price_rupee", "kms_driven_num", "engine_cc", "seats_num", "owner_number", "manufacture"])
data = data[data["price_rupee"] > 0]
data = data[data["kms_driven_num"] >= 0]
data = data[data["engine_cc"] > 0]
data = data[data["seats_num"] > 0]
data = data[data["car_age"] >= 0]

display(data.head())
display(data[["price_rupee", "kms_driven_num", "engine_cc", "seats_num", "owner_number", "manufacture", "car_age"]].describe().T)

### Вывод по очистке

После преобразования задача становится корректной регрессионной задачей: целевая переменная и основные технические характеристики представлены численно. Это важнее, чем просто применить модель к `object`-колонкам, потому что строковые значения с единицами измерения модель не понимает как количество.

## 4. Исследовательский анализ данных

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data["price_rupee"], bins=40, kde=True, ax=axes[0], color="#59A14F")
axes[0].set_title("Распределение цены в рупиях")
axes[0].set_xlabel("Цена, rupee")

sns.histplot(np.log1p(data["price_rupee"]), bins=40, kde=True, ax=axes[1], color="#F28E2B")
axes[1].set_title("Распределение log1p(price)")
axes[1].set_xlabel("log1p(price)")
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = ["price_rupee", "kms_driven_num", "engine_cc", "seats_num", "owner_number", "manufacture", "car_age"]

plt.figure(figsize=(9, 6))
sns.heatmap(data[numeric_cols].corr(), annot=True, fmt=".2f", cmap="vlag", center=0, linewidths=.5)
plt.title("Корреляции числовых признаков")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.scatterplot(data=data, x="manufacture", y="price_rupee", alpha=.35, ax=axes[0])
axes[0].set_title("Год выпуска vs цена")

sns.scatterplot(data=data, x="kms_driven_num", y="price_rupee", alpha=.35, ax=axes[1])
axes[1].set_title("Пробег vs цена")

sns.scatterplot(data=data, x="engine_cc", y="price_rupee", alpha=.35, ax=axes[2])
axes[2].set_title("Объем двигателя vs цена")
plt.tight_layout()
plt.show()

In [ ]:
for col in ["brand", "fuel_type", "transmission", "ownership", "seats_num"]:
    stats = (
        data.groupby(col, dropna=False)["price_rupee"]
        .agg(["count", "median"])
        .sort_values("median", ascending=False)
        .head(15)
        .reset_index()
    )
    plt.figure(figsize=(10, 4))
    sns.barplot(data=stats, y=col, x="median", color="#E15759")
    plt.title(f"Медианная цена по группам: {col}")
    plt.xlabel("Медианная цена, rupee")
    plt.tight_layout()
    plt.show()
    display(stats)

### Выводы EDA

- Распределение цены скошено вправо: дорогих автомобилей мало, но они сильно влияют на среднее.
- Более новые автомобили обычно стоят дороже, поэтому `manufacture` и `car_age` должны быть важными признаками.
- Большой пробег чаще связан с меньшей ценой.
- Бренд, тип топлива и коробка передач создают заметные различия между группами.
- `car_name` в сыром виде содержит слишком много уникальных моделей, поэтому для устойчивой модели лучше использовать `brand`, а не всю строку названия.

## 5. Подготовка признаков

In [ ]:
target = "price_rupee"

feature_cols = [
    "brand",
    "fuel_type",
    "transmission",
    "owner_number",
    "manufacture",
    "car_age",
    "kms_driven_num",
    "engine_cc",
    "seats_num",
]

X = data[feature_cols].copy()
y = data[target].copy()

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Числовые признаки:", numeric_features)
print("Категориальные признаки:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

## 6. Baseline и модели

Baseline нужен как нижняя граница качества. Если сложная модель не лучше медианного прогноза, значит пайплайн построен неправильно или признаки не несут сигнала.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

models = {
    "Baseline median": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", DummyRegressor(strategy="median")),
    ]),
    "RandomForest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "GradientBoosting": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
    ]),
}


def regression_report(name, fitted_model, X_part, y_part):
    pred = fitted_model.predict(X_part)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_part, pred),
        "RMSE": np.sqrt(mean_squared_error(y_part, pred)),
        "R2": r2_score(y_part, pred),
    }


reports = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    reports.append(regression_report(name, pipe, X_test, y_test))

metrics = pd.DataFrame(reports).sort_values("MAE")
metrics_display = metrics.copy()
metrics_display["MAE"] = metrics_display["MAE"].round(0).astype(int)
metrics_display["RMSE"] = metrics_display["RMSE"].round(0).astype(int)
metrics_display["R2"] = metrics_display["R2"].round(3)
display(metrics_display)

best_model_name = metrics.iloc[0]["model"]
best_model = models[best_model_name]
print("Лучшая модель по MAE:", best_model_name)

In [ ]:
cv_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
print(f"CV MAE для {best_model_name}: {-cv_scores.mean():,.0f} +/- {cv_scores.std():,.0f}")

## 7. Анализ ошибок модели

In [ ]:
y_pred = best_model.predict(X_test)
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(x=y_test, y=y_pred, alpha=.45, ax=axes[0], color="#4C78A8")
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], color="black", linestyle="--")
axes[0].set_title("Фактическая цена vs прогноз")
axes[0].set_xlabel("Фактическая цена")
axes[0].set_ylabel("Прогноз")

sns.histplot(residuals, bins=40, kde=True, ax=axes[1], color="#B07AA1")
axes[1].axvline(0, color="black", linestyle="--")
axes[1].set_title("Распределение ошибок")
axes[1].set_xlabel("Факт - прогноз")
plt.tight_layout()
plt.show()

errors = X_test.copy()
errors["actual_price"] = y_test
errors["predicted_price"] = y_pred
errors["abs_error"] = np.abs(errors["actual_price"] - errors["predicted_price"])
errors["relative_error_percent"] = errors["abs_error"] / errors["actual_price"] * 100
display(errors.sort_values("abs_error", ascending=False).head(10))

## 8. Важные и неважные признаки

In [ ]:
def get_encoded_feature_names(fitted_pipeline):
    prep = fitted_pipeline.named_steps["preprocessor"]
    names = list(numeric_features)
    if categorical_features:
        encoder = prep.named_transformers_["cat"].named_steps["onehot"]
        names.extend(encoder.get_feature_names_out(categorical_features).tolist())
    return names


def map_encoded_to_original(encoded_name):
    if encoded_name in numeric_features:
        return encoded_name
    for col in categorical_features:
        if encoded_name.startswith(col + "_"):
            return col
    return encoded_name


model_step = best_model.named_steps["model"]
if hasattr(model_step, "feature_importances_"):
    encoded_importance = pd.DataFrame({
        "encoded_feature": get_encoded_feature_names(best_model),
        "importance": model_step.feature_importances_,
    }).sort_values("importance", ascending=False)

    encoded_importance["original_feature"] = encoded_importance["encoded_feature"].apply(map_encoded_to_original)
    grouped_importance = (
        encoded_importance
        .groupby("original_feature", as_index=False)["importance"]
        .sum()
        .sort_values("importance", ascending=False)
    )
    grouped_importance["importance_percent"] = grouped_importance["importance"] / grouped_importance["importance"].sum() * 100

    display(grouped_importance)
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped_importance, x="importance_percent", y="original_feature", color="#59A14F")
    plt.title("Важность исходных признаков")
    plt.xlabel("Доля важности, %")
    plt.ylabel("Признак")
    plt.tight_layout()
    plt.show()
else:
    grouped_importance = pd.DataFrame()
    print("У выбранной модели нет feature_importances_. Используем permutation importance ниже.")

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(perm_importance)
plt.figure(figsize=(10, 5))
sns.barplot(data=perm_importance, x="importance_mean", y="feature", color="#F28E2B")
plt.title("Permutation importance")
plt.xlabel("Падение качества при перемешивании признака")
plt.ylabel("Признак")
plt.tight_layout()
plt.show()

In [ ]:
if not grouped_importance.empty:
    threshold = grouped_importance["importance_percent"].median()
    important_features = grouped_importance[grouped_importance["importance_percent"] >= threshold]
    weak_features = grouped_importance[grouped_importance["importance_percent"] < threshold]
else:
    threshold = perm_importance["importance_mean"].median()
    important_features = perm_importance[perm_importance["importance_mean"] >= threshold].rename(columns={"feature": "original_feature"})
    weak_features = perm_importance[perm_importance["importance_mean"] < threshold].rename(columns={"feature": "original_feature"})

print("Важные признаки:")
display(important_features)

print("Менее важные признаки:")
display(weak_features)

### Вывод по признакам

Самыми полезными для прогноза обычно становятся:

- `brand` - марка автомобиля отражает класс и рыночное позиционирование;
- `manufacture` / `car_age` - более новые машины обычно дороже;
- `engine_cc` - объем двигателя связан с классом автомобиля;
- `kms_driven_num` - большой пробег снижает стоимость;
- `transmission` и `fuel_type` - влияют на спрос и цену.

Слабые признаки не обязательно удалять сразу. Их стоит убирать, если они дают почти нулевую важность, плохо заполняются или усложняют использование модели без выигрыша в качестве.

## 9. Сохранение модели

In [ ]:
metadata = {
    "target": target,
    "features": feature_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "best_model": best_model_name,
    "metrics": metrics.to_dict(orient="records"),
}

with open(ARTIFACTS_DIR / "car_price_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

if joblib is not None:
    joblib.dump(best_model, ARTIFACTS_DIR / "car_price_model.joblib")
    print("Модель сохранена:", ARTIFACTS_DIR / "car_price_model.joblib")
else:
    print("joblib не установлен, модель не сохранена.")

sample = X_test.head(5).copy()
sample["predicted_price_rupee"] = best_model.predict(X_test.head(5))
display(sample)

## Итог

В этой версии ноутбука исходный анализ доведен до законченного ML-пайплайна. Исправлены проблемы с текстовыми числовыми колонками, использованы корректные регрессионные метрики, добавлены графики ошибок и важность признаков. Модель сохранена как единый `Pipeline`, поэтому ее можно применять к новым данным без ручного one-hot кодирования.